# Email Spam Detection

## Part 1: Text Preprocessing and Bag-of-Words Representation

In this section, raw email text is transformed into numerical features that can be used by machine learning models.

The `emails_1.csv` dataset contains two columns:

- `text`: Email content
- `status`: Email label (`spam` or `ham`)

The preprocessing pipeline includes the following steps:

1. Convert the `status` labels to numerical values:
   - `spam` → `1`
   - `ham` → `0`
2. Remove non-alphabetic characters, including numbers and punctuation.
3. Convert all text to lowercase.
4. Generate Bag-of-Words features using `CountVectorizer`.
5. Retain the 15 most frequent words as input features.

### Expected Output

A DataFrame containing:

- 15 Bag-of-Words features
- The numerical `status` column
- All 10 email samples

In [ ]:
import pandas as pd
import re
import numpy as np
import math
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import warnings

# Suppress FutureWarning messages
warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:
# Load the email dataset

df_email = pd.read_csv("../data/email/emails_1.csv")

In [ ]:
# Encode target labels: spam = 1, ham = 0

df_email.loc[df_email['status'] == 'spam', 'status'] = 1
df_email.loc[df_email['status'] == 'ham', 'status'] = 0
df_email['status']


0    1
1    0
2    1
3    0
4    1
5    1
6    0
7    1
8    0
9    1
Name: status, dtype: object

In [ ]:
df_email["text"] = df_email["text"].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))

In [ ]:
df_email["text"] = df_email["text"].apply(lambda x: x.lower())

In [ ]:
list_email = df_email['text'].to_list()

In [ ]:
# Create Bag-of-Words features using the 15 most frequent words

vectorizer = CountVectorizer(max_features=15)  
X = vectorizer.fit_transform(df_email["text"])

# Convert the sparse feature matrix to a DataFrame
word_counts = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# Add the numerical target labels to the feature DataFrame
word_counts['status'] = df_email['status']
word_counts

,and,at,by,claim,enjoy,for,free,get,our,please,project,the,to,you,your,status
0,0,0,0,1,0,0,2,0,1,0,0,0,1,1,1,1
1,1,1,0,0,0,0,0,0,0,1,2,1,0,0,1,0
2,0,0,0,0,1,2,1,1,1,0,0,0,0,1,0,1
3,1,0,1,0,0,0,0,0,0,1,0,1,0,0,1,0
4,0,0,0,1,1,1,1,0,0,0,0,0,0,0,2,1
5,0,1,0,0,0,0,1,1,1,0,0,0,0,0,0,1
6,0,0,0,0,0,1,0,0,0,1,1,1,0,0,0,0
7,0,0,0,1,0,0,1,0,0,0,0,1,1,0,2,1
8,1,0,1,0,0,1,0,1,0,0,0,1,1,1,1,0
9,0,0,0,0,0,1,0,0,0,0,0,0,1,0,2,1


## Part 2: Loading the Preprocessed Email Dataset

From this point onward, the `emails_1.csv` dataset and its derived DataFrame will no longer be used.

The `emails_2.csv` dataset contains the preprocessed representation of 5,172 real emails. Each row represents one email, while each feature column represents the frequency of one of the 3,000 most frequent words in the dataset.

The machine learning models implemented in the following sections will be trained and evaluated using this dataset.

In this section, the dataset is loaded and its first few rows are displayed to examine its structure.

### Expected Output

Display the first five rows of the `emails_2.csv` DataFrame.

In [ ]:
df_email2 = pd.read_csv(r'../data/email/emails_2.csv')

In [ ]:
df_email2.head()

,Email No.,the,to,ect,and,for,of,a,you,hou,...,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry,Prediction
0,Email 1,0,0,1,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Email 2,8,13,24,6,6,2,102,1,27,...,0,0,0,0,0,0,0,1,0,0
2,Email 3,0,0,1,0,0,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Email 4,0,5,22,0,5,1,51,2,10,...,0,0,0,0,0,0,0,0,0,0
4,Email 5,7,6,17,1,5,2,57,0,9,...,0,0,0,0,0,0,0,1,0,0


## Part 3: Train–Test Split

Split the dataset into training and test sets using the following proportions:

- 80% for training
- 20% for testing

After splitting the data, display the number of samples in the training and test sets.

### Expected Output

The following variables should be created:

- `X_train`
- `X_test`
- `y_train`
- `y_test`

Also display the number of samples in `X_train` and `X_test`.

In [ ]:
# Separate features and target labels

X = df_email2.drop('Prediction', axis=1)
X = X.drop(X.columns[0], axis=1)

In [ ]:
y = df_email2['Prediction']

In [ ]:
indices = np.arange(len(df_email2))

In [ ]:
# Shuffle sample indices for a reproducible train-test split

np.random.seed(42)
np.random.shuffle(indices)

In [ ]:
split = int(0.8 * len(df_email2))
train_idx, test_idx = indices[:split], indices[split:]

In [ ]:
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

In [ ]:
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [17]:
X_train

,the,to,ect,and,for,of,a,you,hou,in,...,enhancements,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry
1566,1,1,1,0,1,0,5,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1988,13,20,66,8,10,0,195,4,33,16,...,0,0,0,0,0,0,0,0,0,0
1235,1,14,3,6,2,2,238,0,0,41,...,0,0,1,0,1,0,0,0,5,0
3276,0,0,1,0,1,0,4,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3438,11,6,2,1,5,6,93,3,1,22,...,0,0,0,0,0,0,0,0,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1301,2,2,2,1,1,0,10,0,0,1,...,0,0,0,0,0,0,0,0,1,0
3430,24,18,23,9,7,6,125,4,10,33,...,0,0,0,0,0,0,0,0,0,0
1304,5,5,4,1,4,0,32,0,0,9,...,0,0,0,0,0,0,0,0,14,0
236,5,0,1,1,4,3,20,0,0,3,...,0,0,0,0,0,0,0,0,1,0


In [18]:
X_test

,the,to,ect,and,for,of,a,you,hou,in,...,enhancements,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry
2599,4,3,14,0,8,2,41,2,6,2,...,0,0,0,0,0,0,0,0,0,0
2573,8,3,2,4,1,1,29,1,0,2,...,0,0,0,0,0,0,0,0,0,0
3280,5,3,1,7,4,4,58,5,0,10,...,0,0,0,0,0,0,0,0,2,0
3186,1,1,7,0,5,1,27,1,3,1,...,0,0,0,0,0,0,0,0,0,0
3953,16,11,1,11,1,8,94,6,3,24,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,0,3,2,0,2,0,16,0,0,1,...,0,0,0,0,0,0,0,0,0,0
466,4,5,1,0,5,1,28,1,0,1,...,0,0,0,0,0,0,0,0,0,0
3092,0,0,1,0,1,0,5,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3772,2,11,1,6,4,5,58,10,0,12,...,0,0,0,0,0,0,0,0,3,0


In [19]:
y_train

1566    0
1988    0
1235    1
3276    0
3438    0
       ..
1301    0
3430    0
1304    0
236     0
4323    0
Name: Prediction, Length: 4137, dtype: int64

In [20]:
y_test

2599    0
2573    0
3280    1
3186    0
3953    1
       ..
4426    0
466     0
3092    0
3772    1
860     1
Name: Prediction, Length: 1035, dtype: int64

In [21]:
print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 4137
Test size: 1035


## Part 4: Evaluation Metrics

Before implementing and training the classification models, several evaluation metrics are defined for binary classification.

The following functions should be implemented from scratch:

- `accuracy(y_true, y_pred)`: Proportion of correctly classified samples
- `precision(y_true, y_pred)`: Proportion of predicted positive samples that are actually positive
- `recall(y_true, y_pred)`: Proportion of actual positive samples that are correctly identified
- `f1_score(y_true, y_pred)`: Harmonic mean of precision and recall

Each function should return the numerical value of the corresponding metric.

Use the following sample inputs to test the implementations:

```python
y_true = [0, 1, 1, 0, 1]
y_pred = [0, 1, 0, 0, 1]
```

### Expected Output

```text
Accuracy:  0.80
Precision: 1.00
Recall:    0.67
F1-score:  0.80
```




---

$$ accuracy = \frac{TP + TN}{TP + TN + FP + FN} $$

In [ ]:
def accuracy(y_true, y_pred):
    """
    Calculate the accuracy score for binary classification.

    Parameters
    ----------
    y_true : array-like
        Ground-truth binary labels.
    y_pred : array-like
        Predicted binary labels.

    Returns
    -------
    float
        The proportion of correctly classified samples.
    """

    FP = 0
    TP = 0
    FN = 0
    TN = 0
    for i in range(0, len(y_pred)):
        if (y_pred[i] == 1):
            if (y_true[i] == 1):
                TP += 1
            
            else:
                FP += 1
                
        else:
            if (y_true[i] == 1):
                FN += 1
            
            else:
                TN += 1
    
    return (TP + TN) / (TP + TN + FP + FN)

In [23]:
a = accuracy([0, 1, 1, 0, 1], [0, 1, 0, 0, 1])
a

0.8

---

$$ precision = \frac{TP}{TP + FP} $$

In [ ]:
def precision(y_true, y_pred):
    """
    Calculate the precision score for binary classification.

    Parameters
    ----------
    y_true : array-like
        Ground-truth binary labels.
    y_pred : array-like
        Predicted binary labels.

    Returns
    -------
    float
        The proportion of predicted positive samples that are truly positive.
    """

    FP = 0
    TP = 0
    for i in range(0, len(y_pred)):
        if (y_pred[i] == 1):
            if (y_true[i] == 1):
                TP += 1
            
            else:
                FP += 1
    
    return (TP) / (TP + FP)

In [25]:
p = precision([0, 1, 1, 0, 1], [0, 1, 0, 0, 1])
p

1.0

---

$$ recall = \frac{TP}{TP + FN} $$

In [ ]:
def recall(y_true, y_pred):
    """
    Calculate the recall score for binary classification.

    Parameters
    ----------
    y_true : array-like
        Ground-truth binary labels.
    y_pred : array-like
        Predicted binary labels.

    Returns
    -------
    float
        The proportion of actual positive samples that are correctly identified.
    """

    TP = 0
    FN = 0
    for i in range(0, len(y_pred)):
        if (y_pred[i] == 1):
            if (y_true[i] == 1):
                TP += 1
        
        else:
            if (y_true[i] == 1):
                FN += 1
                
    return (TP) / (TP + FN)

In [27]:
r = recall([0, 1, 1, 0, 1], [0, 1, 0, 0, 1])
r

0.6666666666666666

---

$$
F_1 = \frac{2 \times precision \times recall}{precision + recall}
$$

In [ ]:
def f1_score1(y_true, y_pred):
    """
    Calculate the F1-score for binary classification.

    Parameters
    ----------
    y_true : array-like
        Ground-truth binary labels.
    y_pred : array-like
        Predicted binary labels.

    Returns
    -------
    float
        The harmonic mean of precision and recall.
    """

    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return (2 * p * r) / (p + r)

---

## Part 5: Logistic Regression Implementation

Implement a Logistic Regression classifier from scratch without using a prebuilt machine learning implementation.

The model should be trained on the training set and evaluated on the test set using the evaluation metrics implemented in the previous section.

You may choose suitable hyperparameters, including:

- Learning rate
- Number of training epochs
- Batch size, if applicable

Feature normalization may be applied before training.

### Requirements

- Implement the Logistic Regression model from scratch
- Train the model using `X_train` and `y_train`
- Generate predictions for `X_test`
- Evaluate the predictions using the manually implemented metrics
- Report accuracy, precision, recall, and F1-score

### Expected Output

Display the following evaluation results on the test set:

```text
Accuracy:
Precision:
Recall:
F1-score:

---

$$ x\_normalized = \frac{x - \mu}{\sigma} $$

In [ ]:
class ZScore:
    """
    Standardize features using statistics calculated from the training data.
    """

    def __init__(self):
        self.means = None
        self.standard_deviations = None

    def fit(self, df):
        """
        Calculate the mean and standard deviation of each training feature.

        Parameters
        ----------
        df : pandas.DataFrame
            Training features.

        Returns
        -------
        ZScore
            The fitted normalizer.
        """
        data = df.to_numpy(dtype=float)

        self.means = np.zeros(data.shape[1])
        self.standard_deviations = np.zeros(data.shape[1])

        for column_index in range(data.shape[1]):
            column = data[:, column_index]

            column_sum = 0
            for value in column:
                column_sum += value

            mean = column_sum / len(column)

            squared_difference_sum = 0
            for value in column:
                squared_difference_sum += (value - mean) ** 2

            standard_deviation = math.sqrt(
                squared_difference_sum / len(column)
            )

            if standard_deviation == 0:
                standard_deviation = 1

            self.means[column_index] = mean
            self.standard_deviations[column_index] = standard_deviation

        return self

    def transform(self, df):
        """
        Normalize data using the training-set statistics.

        Parameters
        ----------
        df : pandas.DataFrame
            Features to be normalized.

        Returns
        -------
        pandas.DataFrame
            Normalized features.
        """
        if self.means is None or self.standard_deviations is None:
            raise ValueError("The normalizer must be fitted before transformation.")

        data = df.to_numpy(dtype=float)
        normalized_data = (
            data - self.means
        ) / self.standard_deviations

        return pd.DataFrame(
            normalized_data,
            index=df.index,
            columns=df.columns
        )

    def fit_transform(self, df):
        """
        Fit the normalizer and transform the training data.

        Parameters
        ----------
        df : pandas.DataFrame
            Training features.

        Returns
        -------
        pandas.DataFrame
            Normalized training features.
        """
        self.fit(df)
        return self.transform(df)

In [30]:
X_train.dtypes

the               int64
to                int64
ect               int64
and               int64
for               int64
                  ...  
infrastructure    int64
military          int64
allowing          int64
ff                int64
dry               int64
Length: 3000, dtype: object

In [ ]:
normalizer = ZScore()

X_train_normalized = normalizer.fit_transform(X_train)
X_test_normalized = normalizer.transform(X_test)

In [ ]:
class LogisticRegression1:
    """
    Implement binary Logistic Regression using mini-batch gradient descent.

    Parameters
    ----------
    eta : float
        Learning rate used to update the model parameters.
    epochs : int
        Number of complete training iterations.
    data : pandas.DataFrame
        Training feature matrix.
    label : array-like
        Binary target labels.
    batch : int, default=40
        Number of samples used in each mini-batch.
    """
    
    def __init__(self, eta, epoch, data, label, batch=40) -> None:
        self.eta = eta
        self.epoch = epoch
        self.weight = np.zeros(data.shape[1]).T
        self.baias = 0
        self.data = data
        self.batch = batch
        self.label = label
        self.conver_pd_to_np_func()
    
    def conver_pd_to_np_func(self):
        """
        Convert the training features from a DataFrame to a NumPy array.

        Non-numeric values are converted to missing values.

        Returns
        -------
        None
        """
        
        self.data_np = self.data.apply(pd.to_numeric, errors='coerce')
        self.data_np = self.data_np.to_numpy()
    
    def train(self):
        """
        Train the Logistic Regression model using mini-batch gradient descent.

        Returns
        -------
        weight : numpy.ndarray
            Learned feature weights.
        bias : float
            Learned bias value.
        """

        n_samples = self.data_np.shape[0]
        n_batches = math.ceil(n_samples / self.batch)
        
        for run in range(self.epoch):
            for i in range(n_batches):
                start = i * self.batch
                end = min(start + self.batch, n_samples)
                
                X_batch = self.data_np[start:end]
                y_batch = self.label[start:end]
                
                self.y_hat = self.y_hat_func(X_batch, self.baias)
                self.prob = self.sigmond_func(self.y_hat)
                
                self.cost_weight, self.cost_baias = self.cost_func(len(y_batch), X_batch, y_batch, self.prob)
                self.weight, self.baias = self.update_parameters(self.weight, self.eta, self.cost_weight, self.baias, self.cost_baias)
        
        return self.weight, self.baias
    
    def y_hat_func(self, X, baias):
        """
        Calculate the linear combination of features and model parameters.

        Parameters
        ----------
        X : numpy.ndarray
            Input feature matrix.

        Returns
        -------
        numpy.ndarray
            Linear model outputs before applying the sigmoid function.
        """

        return np.dot(X, self.weight) + baias
    
    # tabdile y_hat be adadi bine 0 va 1 ba estefade az sigmond         
    def sigmond_func(self, y_hat):
        """
        Convert linear outputs into probabilities using the sigmoid function.

        Parameters
        ----------
        y_hat : numpy.ndarray
            Linear model outputs.

        Returns
        -------
        numpy.ndarray
            Predicted probabilities between 0 and 1.
        """
        
        return (1) / (1 +   np.exp(-y_hat))
    
    # hesab kardane cost 
    def cost_func(self, m, X, y, prob):
        """
        Calculate the gradients of the weights and bias.

        Parameters
        ----------
        m : int
            Number of samples in the current mini-batch.
        X : numpy.ndarray
            Mini-batch feature matrix.
        y : numpy.ndarray
            True labels of the mini-batch.
        probabilities : numpy.ndarray
            Predicted probabilities for the mini-batch.

        Returns
        -------
        weight_gradient : numpy.ndarray
            Gradient of the loss with respect to the weights.
        bias_gradient : float
            Gradient of the loss with respect to the bias.
        """
        
        return np.dot((prob - y).T, X) / m, np.sum(prob - y) / m
    
    # update parametr ha
    def update_parameters(self, weight, eta, cost_weight, baias, cost_baias):
        """
        Update the model parameters using gradient descent.

        Parameters
        ----------
        weight_gradient : numpy.ndarray
            Gradient of the weights.
        bias_gradient : float
            Gradient of the bias.

        Returns
        -------
        weight : numpy.ndarray
            Updated feature weights.
        bias : float
            Updated bias value.
        """
        
        return weight - (eta * cost_weight), baias - (eta * cost_baias)
        
    
            
            
    

In [ ]:
# Train the Logistic Regression model

l = LogisticRegression1(0.015, 2000, X_train_normalized, y_train, 100)
new_weight, new_baias = l.train()

In [ ]:
# Generate probabilities and binary predictions for the test set

y_hat_prec = np.dot(X_test_normalized, new_weight) + new_baias

In [ ]:
test_probabilities = 1 / (1 + np.exp(-y_hat_prec))
prob_y_hat = (test_probabilities >= 0.5).astype(int)

In [ ]:
y_test_np = y_test.to_numpy()

In [39]:
print("######### Logistic Regression #########")

print()
print("Accuracy: ", accuracy(y_test_np, prob_y_hat))
print("Precision: ", precision(y_test_np, prob_y_hat))
print("Recall: ", recall(y_test_np, prob_y_hat))
print("f1_score: ", f1_score1(y_test_np, prob_y_hat))
print()

print("#######################################")

######### Logistic Regression #########

Accuracy:  0.9526570048309179
Precision:  0.9174603174603174
Recall:  0.9262820512820513
f1_score:  0.9218500797448166

#######################################


## Part 6: Multinomial Naive Bayes Implementation

Implement a Multinomial Naive Bayes classifier from scratch without using a prebuilt machine learning implementation.

Train the model using the training set, generate predictions for the test set, and evaluate its performance using the metrics implemented earlier.

### Requirements

- Implement Multinomial Naive Bayes from scratch
- Estimate class prior probabilities
- Estimate feature likelihoods for each class
- Apply Laplace smoothing to avoid zero probabilities
- Use log probabilities to prevent numerical underflow
- Train the model using `X_train` and `y_train`
- Generate predictions for `X_test`
- Report accuracy, precision, recall, and F1-score

### Why Multinomial Naive Bayes?

Multinomial Naive Bayes is suitable for text classification because the input features represent word occurrence counts or frequencies.

Gaussian Naive Bayes assumes that feature values follow a Gaussian distribution. This assumption is generally inappropriate for discrete word-count features.

Other Naive Bayes variants may be used depending on the feature representation:

- **Bernoulli Naive Bayes** is suitable when features indicate whether a word is present or absent.
- **Gaussian Naive Bayes** may be suitable for continuous features that approximately follow a normal distribution.
- **Multinomial Naive Bayes** is commonly used for count-based text features.

### Expected Output

Display the following evaluation results on the test set:

```text
Accuracy:
Precision:
Recall:
F1-score:

In [40]:
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [ ]:
class MultinomialNaiveBayes:
    """
    Implement a Multinomial Naive Bayes classifier for binary text
    classification.

    Parameters
    ----------
    data : pandas.DataFrame
        Training features containing word-count values.
    label : pandas.Series
        Binary target labels.
    """
    
    def __init__(self, data, label) -> None:
        self.data = data
        self.label = label
        
    # mohasebe ehtemal pishin
    def prior_prob(self):
        """
        Calculate the prior probability of each class.

        Returns
        -------
        positive_probability : float
            Prior probability of class 1.
        negative_probability : float
            Prior probability of class 0.
        """
        
        return (len(self.label[self.label == 1])) / self.label.shape[0], (len(self.label[self.label == 0])) / self.label.shape[0]
    
    
    # class 0 va class 1 ra joda mikonim
    def jodasazie_data_class_ha(self):
        """
        Separate training samples according to their class labels.

        Returns
        -------
        data_class_positive : pandas.DataFrame
            Training samples belonging to class 1.
        data_class_negative : pandas.DataFrame
            Training samples belonging to class 0.
        """
        
        return self.data[self.label == 1], self.data[self.label == 0]

    # mohasebe ehtemale har kalame dar class haye 0 va 1
    def word_per_total_with_laplace(self):
        """
        Calculate word likelihoods for both classes using Laplace smoothing.

        Returns
        -------
        dict_prob_positive : dict
            Conditional probability of each word given class 1.
        dict_prob_negative : dict
            Conditional probability of each word given class 0.
        """
        
        self.tedad_kol_kalamate_class_positive = self.data_class_positive.to_numpy().sum()
        self.tedad_kol_kalamate_class_negative = self.data_class_negative.to_numpy().sum()
        dict_prob_positive = {}
        dict_prob_negative = {}
        for word in self.data_class_positive:
            tedade_kalame = self.data_class_positive[word].sum()
            dict_prob_positive[word] = (tedade_kalame + 1) / (self.data.shape[1] + self.tedad_kol_kalamate_class_positive)
            
        for word in self.data_class_negative:
            tedade_kalame = self.data_class_negative[word].sum()
            dict_prob_negative[word] = (tedade_kalame + 1) / (self.data.shape[1] + self.tedad_kol_kalamate_class_negative)

        return dict_prob_positive, dict_prob_negative
    
    # amozeshe model 
    def train(self):
        """
        Estimate class priors and word likelihoods from the training data.

        Returns
        -------
        None
        """
        
        self.positive_class_prob, self.negative_class_prob = self.prior_prob()
        self.data_class_positive, self.data_class_negative = self.jodasazie_data_class_ha()
        self.dict_prob_positive, self.dict_prob_negative = self.word_per_total_with_laplace()
    
    # pishbini y_prec
    def predicte(self, X_test):
        """
        Predict binary class labels for test samples.

        Log probabilities are used to prevent numerical underflow.

        Parameters
        ----------
        X_test : pandas.DataFrame
            Test features containing word-count values.

        Returns
        -------
        numpy.ndarray
            Predicted binary class labels.
        """
        
        lst_prediction = []
        for row in range(X_test.shape[0]):
            column = X_test.columns
            sentence = X_test.iloc[row]
            log_prob_positive = np.log10(self.positive_class_prob)
            log_prob_negative = np.log10(self.negative_class_prob)
            for word in column:
                if (sentence[word] != 0):
                    log_prob_positive += (sentence[word] * np.log10(self.dict_prob_positive[word]))
                    log_prob_negative += (sentence[word] * np.log10(self.dict_prob_negative[word]))
            
            if (log_prob_positive > log_prob_negative):
                lst_prediction.append(1)
            else: 
                lst_prediction.append(0)
        return np.array(lst_prediction)      
    
        

In [ ]:
# Train the Multinomial Naive Bayes model

m = MultinomialNaiveBayes(X_train, y_train)
m.train()

In [43]:
y_pred = m.predicte(X_test)

In [44]:
y_test2 = y_test.to_numpy()

In [45]:
print("######### Multinomial Naive Bayes #########")

print()
print("Accuracy: ", accuracy(y_test2, y_pred))
print("Precision: ", precision(y_test2, y_pred))
print("Recall: ", recall(y_test2, y_pred))
print("f1_score: ", f1_score1(y_test2, y_pred))
print()

print("#######################################")

######### Multinomial Naive Bayes #########

Accuracy:  0.9410628019323671
Precision:  0.8702064896755162
Recall:  0.9455128205128205
f1_score:  0.9062980030721965

#######################################
